# 02_silver_crosswalk_clean

## Purpose
Transform Bronze crosswalk/reference tables to Silver layer:
- `cities_crosswalk` - City to County/State mapping
- `county_crosswalk` - County to Metro/CBSA mapping
- Create unified region dimension table

## Transformations
- Standardize column names
- Clean and validate data
- Join crosswalks to create comprehensive region dimension

In [0]:
%run "./common/udf_library"

In [0]:
from pyspark.sql import functions as F
import uuid

CAT = "zillow"
BRONZE = "zillow_bronze"
SILVER = "zillow_silver"

run_id = str(uuid.uuid4())

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CAT}.{SILVER}")

print(f"Run ID: {run_id}")

In [0]:
# Transform cities_crosswalk
print("="*60)
print("Processing: cities_crosswalk")
print("="*60)

try:
    df_cities = spark.table(f"{CAT}.{BRONZE}.cities_crosswalk_bronze")
    
    # Drop audit columns from bronze
    for col in ['load_dt', 'source_path', 'ingest_mode']:
        if col in df_cities.columns:
            df_cities = df_cities.drop(col)
    
    # Standardize column names
    df_cities = standardize_dataframe_columns(df_cities)
    
    # Clean data
    df_cities = (df_cities
        .withColumn("city", F.trim(F.col("city")))
        .withColumn("county", F.trim(F.col("county")))
        .withColumn("state", F.trim(F.upper(F.col("state"))))
        .filter(F.col("city").isNotNull() & (F.col("city") != ""))
    )
    
    # Deduplicate
    before = df_cities.count()
    df_cities = df_cities.dropDuplicates(["unique_city_id"])
    after = df_cities.count()
    
    # Add audit columns
    df_cities = (df_cities
        .withColumn("processed_dt", F.current_timestamp())
        .withColumn("silver_run_id", F.lit(run_id))
    )
    
    # Write to Silver
    target = f"{CAT}.{SILVER}.cities_crosswalk_silver"
    df_cities.write.format("delta").mode("overwrite").saveAsTable(target)
    
    spark.sql(f"COMMENT ON TABLE {target} IS 'Silver: City to County/State crosswalk, cleaned and standardized'")
    
    print(f"Rows: {before} -> {after} (removed {before-after} duplicates)")
    print(f"Written to: {target}")
    display(df_cities.limit(5))
except Exception as e:
    print(f"ERROR: {e}")

In [0]:
# Transform county_crosswalk
print("\n" + "="*60)
print("Processing: county_crosswalk")
print("="*60)

try:
    df_county = spark.table(f"{CAT}.{BRONZE}.county_crosswalk_bronze")
    
    # Drop audit columns
    for col in ['load_dt', 'source_path', 'ingest_mode']:
        if col in df_county.columns:
            df_county = df_county.drop(col)
    
    # Standardize column names
    df_county = standardize_dataframe_columns(df_county)
    
    # Clean data
    df_county = (df_county
        .withColumn("county_name", F.trim(F.col("county_name")))
        .withColumn("state_name", F.trim(F.col("state_name")))
        .withColumn("metro_name_zillow", F.trim(F.col("metro_name_zillow")))
        .withColumn("cbsa_name", F.trim(F.col("cbsa_name")))
        # Ensure FIPS codes are properly formatted
        .withColumn("fips", F.lpad(F.col("fips").cast("string"), 5, "0"))
        .withColumn("state_fips", F.lpad(F.col("state_fips").cast("string"), 2, "0"))
        .withColumn("county_fips", F.lpad(F.col("county_fips").cast("string"), 3, "0"))
    )
    
    # Deduplicate
    before = df_county.count()
    df_county = df_county.dropDuplicates(["fips"])
    after = df_county.count()
    
    # Add audit columns
    df_county = (df_county
        .withColumn("processed_dt", F.current_timestamp())
        .withColumn("silver_run_id", F.lit(run_id))
    )
    
    # Write to Silver
    target = f"{CAT}.{SILVER}.county_crosswalk_silver"
    df_county.write.format("delta").mode("overwrite").saveAsTable(target)
    
    spark.sql(f"COMMENT ON TABLE {target} IS 'Silver: County to Metro/CBSA crosswalk with standardized FIPS codes'")
    
    print(f"Rows: {before} -> {after} (removed {before-after} duplicates)")
    print(f"Written to: {target}")
    display(df_county.limit(5))
except Exception as e:
    print(f"ERROR: {e}")

In [0]:
# Create unified region dimension by joining crosswalks
print("\n" + "="*60)
print("Creating: dim_region (unified region dimension)")
print("="*60)

try:
    df_cities = spark.table(f"{CAT}.{SILVER}.cities_crosswalk_silver")
    df_county = spark.table(f"{CAT}.{SILVER}.county_crosswalk_silver")
    
    # Join city to county crosswalk
    df_dim = (df_cities
        .join(
            df_county.select(
                F.col("county_name").alias("county_join"),
                F.col("state_name").alias("state_join"),
                "fips",
                "state_fips",
                "county_fips",
                "metro_name_zillow",
                "cbsa_name",
                "cbsa_code",
                "county_region_id_zillow",
                "metro_region_id_zillow"
            ),
            on=[
                F.lower(df_cities.county) == F.lower(F.col("county_join")),
                F.lower(df_cities.state) == F.lower(F.col("state_join"))
            ],
            how="left"
        )
        .drop("county_join", "state_join")
    )
    
    # Add region level column
    df_dim = df_dim.withColumn("region_level", F.lit("city"))
    
    # Add surrogate key
    df_dim = df_dim.withColumn(
        "region_key",
        F.md5(F.concat_ws("|", F.col("unique_city_id"), F.col("city"), F.col("state")))
    )
    
    # Reorder columns
    df_dim = df_dim.select(
        "region_key",
        "region_level",
        "unique_city_id",
        "city",
        "county",
        "state",
        "fips",
        "state_fips",
        "county_fips",
        "metro_name_zillow",
        "cbsa_name",
        "cbsa_code",
        "county_region_id_zillow",
        "metro_region_id_zillow",
        "processed_dt",
        "silver_run_id"
    )
    
    # Write dimension table
    target = f"{CAT}.{SILVER}.dim_region"
    df_dim.write.format("delta").mode("overwrite").saveAsTable(target)
    
    spark.sql(f"COMMENT ON TABLE {target} IS 'Dimension: Unified region hierarchy with city/county/metro/state linkage'")
    
    print(f"Created dimension with {df_dim.count():,} rows")
    print(f"Written to: {target}")
    
    # Show sample with joins
    display(df_dim.filter(F.col("metro_name_zillow").isNotNull()).limit(10))
    
except Exception as e:
    print(f"ERROR: {e}")

In [0]:
# Summary
print("\n" + "="*60)
print("CROSSWALK TRANSFORMATION SUMMARY")
print("="*60)

display(spark.sql(f"""
    SELECT table_name, 
           (SELECT COUNT(*) FROM {CAT}.{SILVER}.cities_crosswalk_silver) as cities_rows,
           (SELECT COUNT(*) FROM {CAT}.{SILVER}.county_crosswalk_silver) as county_rows,
           (SELECT COUNT(*) FROM {CAT}.{SILVER}.dim_region) as dim_region_rows
    FROM (SELECT 'Summary' as table_name)
"""))